# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields (by `@id`). This allows us to reference all data programmatically by their Croissant `@id`.

**Note:** In Croissant, 'record sets' correspond to primary data tables. Fields/columns have their own Croissant `@id` and must be referenced by it.

In [ ]:
# List all available record sets by their `@id` and name

if hasattr(metadata, 'record_sets') and metadata.record_sets:
    print('Available Record Sets:')
    for rs in metadata.record_sets:
        print(f"- @id: {rs.id} | name: {getattr(rs, 'name', '[no name]')}")
        if hasattr(rs, 'fields'):
            print("  Fields:")
            for f in rs.fields:
                print(f"    - @id: {f.id} | name: {getattr(f, 'name', '[no name]')} | type: {getattr(f, 'data_type', '[no type]')}")
else:
    print("No record sets listed in the Croissant metadata. Attempting automatic inference by inspecting records...")

# mlcroissant provides a dataset.record_sets() function to list all available sets (if the Croissant provides them)
available_record_sets = dataset.record_sets()
if available_record_sets:
    print('Record sets found by dataset.record_sets():')
    pprint.pprint(available_record_sets)
else:
    print("No record sets found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

**Important:** All entities must be referenced by their `@id` in code.

We'll extract the first available record set and display its records.

In [ ]:
# List record sets for reference
record_set_ids = available_record_sets
print('RecordSet @ids:', record_set_ids)


# If record_set_ids is empty, stop here
if not record_set_ids:
    raise ValueError('No record sets detected in this Croissant schema.')

# We'll extract the first record set (@id)
main_record_set_id = record_set_ids[0]

# Extract all records from the main record set
records = list(dataset.records(record_set=main_record_set_id))

df = pd.DataFrame(records)
print('Columns:', df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)
We'll process the data as follows:
- Select a numeric field (by its Croissant `@id` as found in the DataFrame columns).
- Filter records based on a threshold for that field.
- Normalize the numeric field for filtered records.
- Optionally, group by a categorical field (referenced by its `@id`).

Update the variables below according to the `@id`s from the DataFrame columns.

In [ ]:
# Set the correct @id for a numeric field (change after inspecting df.columns)
# For example, if there is a field 'cr:field/age', set:
# numeric_field_id = 'cr:field/age'

numeric_fields = [c for c in df.columns if df[c].dtype in ['int64','float64']]
if not numeric_fields:
    # Try to heuristically convert a likely column
    for c in df.columns:
        try:
            df[c] = pd.to_numeric(df[c])
            if df[c].dtype in ['int64','float64']:
                numeric_fields.append(c)
        except Exception:
            continue
print('Numeric candidate fields by @id:', numeric_fields)
# Pick the first numeric field found as default
numeric_field_id = numeric_fields[0] if numeric_fields else df.columns[0]
print('Using numeric field:', numeric_field_id)

threshold = 10 # example threshold, adjust as appropriate
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Choose a group field (must exist in df.columns)
categorical_candidates = [c for c in df.columns if df[c].dtype == 'object' and c != numeric_field_id]
if categorical_candidates:
    group_field_id = categorical_candidates[0]
    print(f'Grouping by {group_field_id}')
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
    print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
    print(grouped.head())
else:
    print('No categorical field found for grouping!')

## 5. Visualization
Visualize distributions or relationships.

We'll plot (1) the numeric field's distribution after filtering, and (2) if a categorical field is present, a bar plot of the group mean.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id], kde=True, bins=20)
plt.title(f'Distribution of {numeric_field_id} (> {threshold})')
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If a grouped mean was calculated, plot the group means
if 'group_field_id' in locals():
    plt.figure(figsize=(10,4))
    grouped.sort_values(ascending=False).plot.bar()
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to:

- Load Croissant-encoded biomedical data using `mlcroissant`
- Reference all dataset elements by their Croissant `@id`
- Load records into DataFrames for analysis
- Perform basic numerical and categorical analyses
- Visualize tabular data relationships

You can further adapt this notebook for in-depth analysis based on your specific scientific or clinical research questions!